In [35]:
print("====COVID-19 DATA ANALYSIS====")
import numpy as np

np.set_printoptions(suppress=True, precision=2)

countries = np.array([
    "Pakistan", "India", "Brazil", "Uk", "UAE"
])

rng = np.random.default_rng(42)

base_cases = np.array([1000, 2000, 3000, 4000, 5000,])
deaths_cases = np.array([70, 150, 250, 350, 450,])

cases = np.maximum(
    0,
    (
        
        base_cases[:,None] #  # ye function upar base_array ho columns ma lekhy ga
        *(1 + 0.025 * np.arange(21)[None,:]) # # ye 1 sy 20 tk no ka array banye ga pher sb ko 0.025 yani 2.5 sy multiple kary ga aur sb aur decimal 
                                            #sy phely 1add kary ga
        + rng.normal(0, 250,(5,21)) # #matlab 5 random no aye gy aur 0 ky aspas hongy aur 250 jeetny  bary hongy
    ).round()
)

deaths = np.maximum(
    0,
    (
        deaths_cases[:,None]  # ye function upar base_array ho columns ma lekhy ga
        + rng.normal(0, 250, (5,21))  #matlab 5 random no aye gy aur 0 ky aspas hongy aur 250 jeetny  bary hongy
    ).round()
)

cases[0, 4] = np.nan # 0 = pkistan 4 matlab panchwa cloums ko missing krdu
cases[2, 7] = np.nan # 2 = brazil 10 matlab gyira columns ko missing krdu
cases[1, 8] = np.nan # 1 = india 8 matlab nawa columns missing krdu 

data = np.stack([deaths, cases,], axis=2) # ek varaible banaya data ky name sy np.stack means deaths and cases ko mila kr 2d array ban do aur axis=2
                                          #columns ma lekh do

print("Data Shape:", np.shape(data)) # np.shape means data ki shape pata lago
print("Countries:", countries) # countries bhei malom karo

print("===Missing Data Analysis and simulated===")

missing_mask = np.isnan(data) # ek variable banaya missing_mask ly name sy np.isnan = matlab data missing ha

print("Missing values:", np.sum(missing_mask)) # missing values ko sum krwaya aur print krwaya

country_metric_means = np.nanmean(data, axis=1, keepdims=True) # variable banaya np.nanmean = missing values ko chor kr average nikalna 
                                                               #keepdism=true= matlab calculation ky baad bhei array ki dimentions ko true rakhna 
      
clean_data = np.where( # variable banya np.where=if else 
    missing_mask, # variable ko recall krwaya
    country_metric_means, # variable ko recall krwaya
    data     # variable ko recall krwaya
)

print("Missing values after cleaning:", np.isnan(clean_data).sum())# false data ky sahi calculation ko sum krwaya and print krwaya and recall the 
                                                                    #variable

clean_cases = clean_data[:, :, 0] # variable banaya varaible ko recall krwaya and [:, :, 0] = matlab data ki shape maloom ki
clean_deaths =  clean_data[:, :, 1] # variable banaya varaible ko recall krwaya and [:, :, 0] = matlab data ki shape maloom ki

print("====Country wise and day wise anlysis====")

country_total_cases = np.sum(clean_cases, axis=1) # variable banaya np.sum = country ky Total cases ko sum karwaya 
                                                   #varaible recall krwaya and axis=1 kaha ky coulns ma lekh deya
country_average_cases = np.mean(clean_cases, axis=1)# variable banaya np.mean = country ky average cases ko maloom kia varaible recall krwaya and 
                                                    #axis=1 kaha ky coulns ma lekh deya

print("Country Total cases:") # print krwaya 
print(np.column_stack((countries, country_total_cases)))# np.coulumn_stack = matlab coloumns ko mila kr ek array banao and recalls variables

print("Daily totals Acroos Country:")
daily_global_cases =np.sum(clean_cases, axis=0) # variable banaya np.sum = total caes ko add krwaya and again recall a varaibleand axis = matlab 
                                                #rows ma batana 
print(daily_global_cases) # variable ko print krwaya

print("====Country Rolling Average===")

window = 7 # windows ko define kia

rolling_cases = np.stack([# varaible banaya np.stack = matlab hr country ky 7 days calculate karo ko new numpy array banao
    np.mean(clean_cases[:, i : i + window], axis=1)# np.mean = matlab hr country ky 7 din ka average maloom karo 
                                                # := sari countries select karo  i : i= matlab 7 din select karo axis=1 7 day ka average calculate karo
    for i in range(clean_cases.shape[1] - window + 1) # for loop chalayi aur kaha ky clean cases ki row ma sy - krdu countries ky coloumn ko
], axis=1) # 7 din ka average calculate karo

print("Rolling average shape:",rolling_cases.shape) # maloom kia rolling ki average shape
print(" first's Countries rolling average:",rolling_cases[0]) # hr  first's country ky columns ma sy shape maloom karo


print("====Day to day change and percentage change===")

daily_cases = np.diff(clean_cases, axis=1) # varaible banaya np.diff = matlab difference moloom karo hr country ki rows and clean cases ky 
                                        #darmiyan difference molloom karo

percentage_change = np.divide( # variable banaya np.divide = divide krta ha 
    daily_cases, # divide karo daily cases ko
    clean_cases[:,:-1],# hr countries ki akhri rows ko chor do baki ko divide karo
    out=np.zeros_like(daily_cases), # agar divide ni hoskta tu output ma pheley sy 0 rakho
    where=clean_cases[:,:-1] !=0
)* 100 # jo result aye aussey * 100 krdu 

print("Daily change for each country:",daily_cases)# hr country ky daily change ko print krwaya and recall the variable
print(daily_cases) # variable ko print krwaya 

print("Percentage change:", percentage_change) # hr country ki percentage maloom ki and recall the varaible
print(percentage_change) # variable ko print krwaya

print("====Anomaly Dectection using Z_zeros====")

country_mean = np.mean(clean_cases, axis=1, keepdims=True) # variable banaya np.mean = clean cases aur countries ki row ka average maloom kia 
                                                           # keepdims=True ka matlab ma calculation ky baad bhei array ki dimension ko true rakhna
country_std = np.std(clean_cases, axis=1, keepdims=True)# variable banaya np.std = clean cases aur countries ki row ka standarad derivatiom maloom kia 
                                                        # keepdims=True ka matlab ma calculation ky baad bhei array ki dimension ko true rakhna

z_zeros = np.divide(            # z_zeros batata ha ki caese average sey ketni door ha np.divide division krta ha 
    clean_cases - country_mean, # clean cases and country mean ko - karo aur divide karo country std sy       
    country_std,                # divide karo
    out=np.zeros_like(clean_cases), # agar output true na o tu aus ko zero krdu 
    where = country_std !=0
)

anomaly_mask = np.abs(z_zeros) > 2 # anomaly_mask = varaible banaya ye result true ya false ma bata ha np.abs = neg value ko positive banata ha 
                                   # ye bata ha key value 2 sey bari ha ya choti

print("anomalies mask per country:") # print krwaya
print(np.sum(anomaly_mask, axis=1))# anomaly_mask and countries ki row ko np.sum krwaya

for i, country in enumerate(countries): #for loop lagayi and countries ko recall krwaya
       anomaly_days = np.flatnonzero(anomaly_mask[i]) # variabke banaya np.f latnonzeros ka matlab anomaly_mask ky andar 0 ky elawa no batao
       print(country, "Anomalies days indices:", anomaly_days) # country ky anomaly_days maloom keye

print("===Country normalized====")

country_maximiums = np.max( # varaible bnaya and np.max = maiximiums
    clean_cases, axis=1, keepdims=True, # maximiums clean cases maloom keya countries ki maximiums row maloom ki keepdmis matlab calculation ky 
                                        #baad array ki dimensio ko true rakta ha  
)

normalized_cases = np.divide( # varaible banya np.divide = division
    clean_cases, # divide krwaya
    country_maximiums, # divide krwaya
    out=np.zeros_like(clean_cases), #  agr output true na ho tu 0 krdu
    where = country_maximiums!=0
)

print("Country normalized shape:", normalized_cases.shape) # country ki shape maloom ki
print(normalized_cases) # print krwaya varaible ko

print("=====Country ranking====")

ranking = np.argsort(country_total_cases)[::-1]# varaible banaya np.argsort matlab country ky tamama cases ko ascending ma lekhy ga [:,:-1] akhri ky 
                                               #  cases ko chor dy ga baki ko ascending ma kary ga 

for rank, index in enumerate(ranking, start=1): # for loop lagya ranking ko recall krwaya aur start 1 matlab ranking sy start hogi
    print(
        f"{rank}.{countries[index]}:" # recall the variable and index
        f"{country_total_cases[index]:,.0f}" # recall the variable and index
)

print("====Cases and deaths colleration===")
print("Cooleration between caes and deaths cases:")

for i, country in enumerate(countries):# for loop lagayi and recall the variable countries
    
    cooleration = np.corrcoef(# Variable banaya and np.corrcoef = matlab ye clean and deaths ky array ma relation between check kary g 
        clean_cases[i], # recall variable clean cases and i
        clean_deaths[i], # recall variable clean deaths and i
    )[0,1]

    print(f"{country} : {cooleration:.3f}") # recall varaible :.3f ka matlab values ma jeetny no ko bs start ky 3 no batana 

print("===Top 3 days cases counts===")

print("Top three days counts:")

top_three_days = np.argsort( # variable bnaya np.argsort #
    clean_cases, axis=1,       # means claen cases and countries ki rows ko ascending ma karo
)[:,-3:][:, ::-1]              # : = saari rows select karo -3 = akhri ki 3 rows select karo -1 = columns ko reverse krdo

for i, country in enumerate(countries): # for loop lagyi countries ko recall krwaya
    print(
        country, # again recall the varaible
        clean_cases[i, top_three_days [i]] # recalls the varabiles
    )

print("===Week to week comparision===")

first_week = np.mean(clean_cases[:, :7],axis=1) # varaible banaya average maloom clean cases and countries row ky dramiyan average maloom kia   
second_week = np.mean(clean_cases[:,7:14], axis=1) # varaible banaya average maloom clean cases and countries row ky dramiyan average maloom kia
third_week = np.mean(clean_cases[:, 14:21], axis=1)  # varaible banaya average maloom clean cases and countries row ky dramiyan average maloom kia

weekly_means = np.stack( # variable banaya np.stack = 3 week ko combain krley numpy new 3d array bnay ga 
    [first_week, second_week, third_week], # recalls the variable 
    axis=1 #recall variable
)

print("Weekly cases:")
print("Colunms: week 1, week 2, week 3,")
print(weekly_means) # recall variable

print("Differnce between weekly average:")
weekly_change = np.diff(weekly_means, axis=1) # variable bnaya np.diff = matlab weekly means and countries ky darmeyan defrence maloom karo
print(weekly_change) # variable ko recall krwaya 

high_case_days = clean_cases > 5000 # variable banaya aur kaha ky agar clean cases 5000 sy zada ha 
moderate_case_days = ( # variable banaya
    (clean_cases >= 2000) # clean cases 2000 ky barabar ho
    & (clean_cases <= 5000) # clean cases 5000 sy kam ho 
)

print("\n11. High-case observations per country:")
print(np.sum(high_case_days, axis=1)) # high case days and countries ki row ko sum krya np.sum ki madad sy 

print("\nModerate-case observations per country:")
print(np.sum(moderate_case_days, axis=1))  # moderate case days and countries ki row ko sum krya np.sum ki madad sy 

print("\n" + "=" * 50)
print("PROJECT SUMMARY")
print("=" * 50)

print("Total observations:", clean_data.size)
print("Countries analyzed:", len(countries))
print("Days analyzed:", clean_cases.shape[1])
print("Overall mean daily cases:", np.mean(clean_cases))
print("Overall median daily cases:", np.median(clean_cases))
print("Overall standard deviation:", np.std(clean_cases))
print("90th percentile:", np.percentile(clean_cases, 90))


print("=====COVID-19 DATA ANALYSIS COMPLETED=====")







    
     




====COVID-19 DATA ANALYSIS====
Data Shape: (5, 21, 2)
Countries: ['Pakistan' 'India' 'Brazil' 'Uk' 'UAE']
===Missing Data Analysis and simulated===
Missing values: 3
Missing values after cleaning: 0
====Country wise and day wise anlysis====
Country Total cases:
[['Pakistan' '2466.0']
 ['India' '3604.0']
 ['Brazil' '5409.0']
 ['Uk' '7474.0']
 ['UAE' '9375.0']]
Daily totals Acroos Country:
[1924. 1737.  880. 1306. 1104.  396.  823. 1335. 1788. 1210. 1453. 1568.
 2203. 2235. 1191. 1409. 1055. 1775.  975. 1082.  879.]
====Country Rolling Average===
Rolling average shape: (5, 15)
 first's Countries rolling average: [ 86.71  84.    84.    96.57  56.57  74.86  60.   108.71 110.14 110.14
 114.   131.86 172.14 201.29 156.86]
====Day to day change and percentage change===
Daily change for each country: [[ -19.    0.  280. -280.  179.  -50. -129.    0.   88.  -88.  128.  -53.
   395. -460.  -10.  115.   10.  285. -131. -120.]
 [-516.    0.    0.   53.  -53.  309. -215.  -94.    0.  228.  132.  28